# Tutorial 8: Noise Models and Error Mitigation

Running quantum finance algorithms on real hardware introduces noise.
This notebook demonstrates qufin's noise models and error mitigation strategies.

**Strategies covered**: ZNE, TREX, readout calibration, PEC, CDR.

**Reference**: Temme, Bravyi, Gambetta (2017). Czarnik et al. (2021).

In [ ]:
import numpy as np
np.random.seed(42)

## 1. Ideal vs Noisy Simulation

qufin provides 4 device noise profiles modeling real IBM hardware.

In [ ]:
from qufin.backends.qiskit_backend import QiskitAerBackend
from qufin.backends.noise_models import NoisyAerBackend, IBM_EAGLE_R3

ideal_backend = QiskitAerBackend(method="automatic", seed=42)  # shots at run()

# Noisy backend with IBM Eagle r3 device profile
noisy_backend = NoisyAerBackend(profile=IBM_EAGLE_R3, seed=42)

print(f"Ideal backend: {ideal_backend.backend_id}")
print(f"Noisy backend: {noisy_backend.backend_id}")
print(f"Noise profile: {noisy_backend.noise_profile.name}")

## 2. Noise Impact on a Simple Circuit

In [ ]:
from qiskit.circuit import QuantumCircuit

# Bell state circuit
qc = QuantumCircuit(2, 2)
qc.h(0)
qc.cx(0, 1)
qc.measure([0, 1], [0, 1])

ideal_counts = ideal_backend.run(qc, shots=8192).counts
noisy_counts = noisy_backend.run(qc, shots=8192).counts

print(f"Ideal counts: {ideal_counts}")
print(f"Noisy counts: {noisy_counts}")

## 3. Readout Error Calibration

In [ ]:
from qufin.backends.error_mitigation import calibrate_readout, mitigate_readout

# Calibrate the readout error (signature: calibrate_readout(n_qubits, backend, shots)).
cal_matrix = calibrate_readout(2, noisy_backend, shots=8192)

print(f"Calibration matrix shape: {cal_matrix.shape}")
print(f"Diagonal (correct readout probs): {np.diag(cal_matrix).round(4)}")

In [ ]:
# Apply readout mitigation; returns a MitigationResult.
mitigated = mitigate_readout(noisy_counts, cal_matrix, shots=8192)

print(f"Raw noisy:  {noisy_counts}")
print(f"Mitigated:  {mitigated.mitigated_counts}")
print(f"Ideal:      {ideal_counts}")

## 4. Zero-Noise Extrapolation (ZNE)

ZNE runs circuits at increasing noise levels and extrapolates to the zero-noise limit.

In [ ]:
from qufin.backends.error_mitigation import zne_extrapolate

# Observable: probability of the |00> outcome. ZNE calls observable_fn(counts, shots).
def prob_00(counts, shots):
    return counts.get("00", 0) / shots if shots else 0.0

# zne_extrapolate takes a measurement-free circuit (measurements added internally).
qc_zne = QuantumCircuit(2)
qc_zne.h(0)
qc_zne.cx(0, 1)

zne_result = zne_extrapolate(
    qc_zne,
    noisy_backend,
    scale_factors=[1, 3, 5],
    shots=8192,
    observable_fn=prob_00,
)

print(f"ZNE extrapolated value: {zne_result['mitigated_value']:.6f}")
print(f"Raw noisy values:       {[round(v, 4) for v in zne_result['raw_values']]}")

## 5. TREX (Twirled Readout Error eXtinction)

In [ ]:
from qufin.backends.error_mitigation import trex_mitigate

# TREX takes a measurement-free circuit and returns a MitigationResult.
trex_result = trex_mitigate(
    qc_zne,            # reuse the measurement-free Bell circuit
    noisy_backend,
    n_twirls=16,
    shots_per_twirl=1024,
)

print(f"TREX raw counts:       {trex_result.raw_counts}")
print(f"TREX mitigated counts: {trex_result.mitigated_counts}")

## 6. Noise Sweep

Compare results across different noise profiles.

In [ ]:
from qufin.backends.noise_models import IBM_EAGLE_R3, IBM_HERON_R2

profiles = [IBM_EAGLE_R3, IBM_HERON_R2]

for profile in profiles:
    nb = NoisyAerBackend(profile=profile, seed=42)
    counts = nb.run(qc, shots=8192).counts
    # Fidelity: fraction in expected states
    good = counts.get("00", 0) + counts.get("11", 0)
    total = sum(counts.values())
    print(f"  {profile.name:<25s} fidelity={good/total:.4f}")

## 7. Dynamical Decoupling

In [ ]:
from qufin.backends.dynamical_decoupling import insert_dd_sequences, DDConfig, DDSequence

# DD is inserted into idle periods of a measurement-free circuit.
qc_core = QuantumCircuit(2)
qc_core.h(0)
qc_core.cx(0, 1)
qc_core.id(0)  # idle slot where DD can be inserted
qc_core.id(1)

qc_dd = insert_dd_sequences(qc_core, DDConfig(sequence_type=DDSequence.XY4))
qc_dd.measure_all()

dd_counts = noisy_backend.run(qc_dd, shots=8192).counts
good_dd = sum(v for k, v in dd_counts.items() if k.replace(" ", "") in ("00", "11"))
total_dd = sum(dd_counts.values())

print(f"Without DD fidelity: {good/total:.4f}")
print(f"With DD fidelity:    {good_dd/total_dd:.4f}")

## Summary

| Strategy | Overhead | Bias | Best For |
|:---------|:---------|:-----|:---------|
| Readout cal. | Low | Readout only | All circuits |
| ZNE | 3x circuit executions | Gate + readout | Short circuits |
| TREX | 32x randomizations | Readout | High readout error |
| PEC | Exponential sampling | Unbiased | Research |
| CDR | Training circuits | Gate errors | Variational |

**Next**: Tutorial 09 covers running on real IBM Quantum hardware.